In [2]:
#!/usr/bin/env python
from __future__ import annotations

import os
import json
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional, Tuple

import pandas as pd
from sqlalchemy import create_engine, text
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
from pyproj import Transformer
from pystac_client import Client

# ----------------------------------------------------------------------
# Configurações gerais
# ----------------------------------------------------------------------
PC_STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

PG_USER = "postgres"
PG_PASS = "postgres"
PG_HOST = "localhost"
PG_PORT = "5432"
PG_DB   = "geologia"

PG_CONN_STR = (
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)


# ----------------------------------------------------------------------
# Utilitários gerais
# ----------------------------------------------------------------------
def parse_target_date(date_str: str) -> datetime:
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return dt.replace(tzinfo=timezone.utc)


def item_datetime(item) -> datetime:
    """
    Extrai datetime (timezone-aware) de um pystac.Item.
    """
    if getattr(item, "datetime", None) is not None:
        dt = item.datetime
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt

    props = getattr(item, "properties", {}) or {}
    dt_str = props.get("datetime") or props.get("start_datetime")
    if dt_str is None:
        raise RuntimeError(f"Item {getattr(item, 'id', '?')} sem campo datetime.")

    if dt_str.endswith("Z"):
        dt_str = dt_str.replace("Z", "+00:00")
    dt = datetime.fromisoformat(dt_str)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt


def get_pc_client() -> Client:
    return Client.open(PC_STAC_URL)


# ----------------------------------------------------------------------
# Acesso ao PostGIS
# ----------------------------------------------------------------------
def get_folha_geom_geojson(
    codigo_folha: str,
    conn_str: str = PG_CONN_STR,
    schema: str = "carto",
    table: str = "folhas_cartograficas",
) -> Dict[str, Any]:
    """
    Retorna a geometria da folha em WGS84 (EPSG:4326) como GeoJSON.
    """
    engine = create_engine(conn_str)
    sql = text(
        f"""
        SELECT ST_AsGeoJSON(ST_Transform(geom, 4326)) AS geom_json
        FROM {schema}.{table}
        WHERE codigo = :codigo
        LIMIT 1;
        """
    )
    with engine.begin() as conn:
        row = conn.execute(sql, {"codigo": codigo_folha}).fetchone()

    if row is None or row.geom_json is None:
        raise RuntimeError(f"Não foi encontrada geometria para a folha '{codigo_folha}'")

    return json.loads(row.geom_json)


def listar_folhas_25k_sb21_za_ii(
    conn_str: str = PG_CONN_STR,
    prefixo: str = "SB21_ZA_II",
) -> List[str]:
    """
    Lista códigos de folhas em carto.folhas_cartograficas cujo código
    começa com prefixo (SB21_ZA_II) e cuja escala é 25k (heurística via CAST).
    """
    engine = create_engine(conn_str)
    sql = text(
        """
        SELECT codigo
        FROM carto.folhas_cartograficas
        WHERE codigo LIKE :pref
          AND (
                CAST(escala AS text) ILIKE '%25k%'
             OR CAST(escala AS text) ILIKE '%25.000%'
             OR CAST(escala AS text) ILIKE '%25000%'
          )
        ORDER BY codigo;
        """
    )
    with engine.begin() as conn:
        rows = conn.execute(sql, {"pref": f"{prefixo}%"}).fetchall()

    return [r.codigo for r in rows]


# ----------------------------------------------------------------------
# Funções geométricas para cobertura
# ----------------------------------------------------------------------
def build_local_equal_area_transformer(geom_geojson: Dict[str, Any]) -> Transformer:
    geom = shape(geom_geojson)
    lon0, lat0 = geom.centroid.x, geom.centroid.y
    laea_proj = (
        f"+proj=laea +lat_0={lat0} +lon_0={lon0} "
        "+datum=WGS84 +units=m +no_defs"
    )
    return Transformer.from_crs("EPSG:4326", laea_proj, always_xy=True)


def coverage_fraction(
    folha_geom_geojson: Dict[str, Any],
    item_geom_geojson: Dict[str, Any],
    transformer: Transformer,
) -> float:
    folha_geom = shape(folha_geom_geojson)
    item_geom = shape(item_geom_geojson)

    project = transformer.transform
    folha_proj = shp_transform(project, folha_geom)
    item_proj = shp_transform(project, item_geom)

    inter = folha_proj.intersection(item_proj)
    if folha_proj.is_empty or inter.is_empty:
        return 0.0
    return inter.area / folha_proj.area


def get_cloud_cover(item) -> Optional[float]:
    props = getattr(item, "properties", {}) or {}
    for key in ("eo:cloud_cover", "s2:cloud_cover"):
        if key in props and props[key] is not None:
            try:
                return float(props[key])
            except Exception:
                continue
    return None


# ----------------------------------------------------------------------
# Busca ASTER
# ----------------------------------------------------------------------
def search_aster_for_folha(
    codigo_folha: str,
    target_date_str: str,
    collection_id: str = "aster-l1t",
    stac_url: str = PC_STAC_URL,
    search_datetime: str = "2000-01-01/2015-12-31",
    min_coverage: float = 0.90,
    max_cloud: Optional[float] = None,
    max_items: int = 2000,
) -> Tuple[List[Dict[str, Any]], Optional[Dict[str, Any]]]:

    folha_geom_geojson = get_folha_geom_geojson(codigo_folha)
    folha_shape = shape(folha_geom_geojson)
    print(
        f"[ASTER] Folha '{codigo_folha}' carregada. "
        f"Área WGS84 (aprox): {folha_shape.area:.6f} (graus²)"
    )
    transformer = build_local_equal_area_transformer(folha_geom_geojson)

    cat = Client.open(stac_url)
    print(f"[ASTER] Buscando itens em '{collection_id}'...")
    search = cat.search(
        collections=[collection_id],
        intersects=folha_geom_geojson,
        datetime=search_datetime,
        max_items=max_items,
    )
    items = list(search.items())
    print(
        f"[ASTER] Total de itens retornados (antes de filtros locais): {len(items)}"
    )

    if not items:
        return [], None

    target_date = parse_target_date(target_date_str).date()
    candidates: List[Dict[str, Any]] = []

    for item in items:
        dt = item_datetime(item)
        dt_date = dt.date()
        delta_days = abs((dt_date - target_date).days)

        cloud = get_cloud_cover(item)
        if cloud is None:
            continue
        if (max_cloud is not None) and (cloud >= max_cloud):
            continue

        frac = coverage_fraction(
            folha_geom_geojson=folha_geom_geojson,
            item_geom_geojson=item.geometry,
            transformer=transformer,
        )
        if frac < min_coverage:
            continue

        candidates.append(
            {
                "id": item.id,
                "datetime": dt.isoformat(),
                "delta_days": delta_days,
                "coverage_fraction": frac,
                "coverage_percent": frac * 100.0,
                "cloud_cover": cloud,
                "item": item,
            }
        )

    if not candidates:
        print(
            "[ASTER] Nenhum item atendeu a:\n"
            f"  coverage >= {min_coverage*100:.1f}%"
            + (f" e cloud < {max_cloud:.2f}%." if max_cloud is not None else ".")
        )
        return [], None

    candidates.sort(
        key=lambda d: (d["delta_days"], d["cloud_cover"], -d["coverage_fraction"])
    )

    print("\n[ASTER] Itens candidatos (ordenados):")
    for c in candidates:
        print(
            f"  id={c['id']}, "
            f"datetime={c['datetime']}, "
            f"delta_days={c['delta_days']}, "
            f"coverage={c['coverage_percent']:.2f}%, "
            f"cloud={c['cloud_cover']:.2f}%"
        )

    best = candidates[0]
    print(
        "\n[ASTER] Melhor item:\n"
        f"  id={best['id']}\n"
        f"  datetime={best['datetime']}\n"
        f"  delta_days={best['delta_days']}\n"
        f"  coverage={best['coverage_percent']:.2f}%\n"
        f"  cloud={best['cloud_cover']:.2f}%"
    )
    return candidates, best


# ----------------------------------------------------------------------
# Busca Sentinel-2
# ----------------------------------------------------------------------
def search_sentinel2_for_folha(
    codigo_folha: str,
    reference_datetime: datetime,
    collection_id: str = "sentinel-2-l2a",
    stac_url: str = PC_STAC_URL,
    search_datetime: str = "2015-06-23/2100-01-01",
    min_coverage: float = 0.90,
    max_cloud: float = 1.0,
    max_items: int = 2000,
    max_print_candidates: int = 0,  # 0 = não listar; >0 = listar primeiros N
) -> Tuple[List[Dict[str, Any]], Optional[Dict[str, Any]]]:

    folha_geom_geojson = get_folha_geom_geojson(codigo_folha)
    folha_shape = shape(folha_geom_geojson)
    print(
        f"[S2] Folha '{codigo_folha}' carregada. "
        f"Área WGS84 (aprox): {folha_shape.area:.6f} (graus²)"
    )
    transformer = build_local_equal_area_transformer(folha_geom_geojson)
    cat = Client.open(stac_url)

    print(f"[S2] Buscando itens em '{collection_id}'...")
    search = cat.search(
        collections=[collection_id],
        intersects=folha_geom_geojson,
        datetime=search_datetime,
        max_items=max_items,
    )
    items = list(search.items())
    print(
        f"[S2] Total de itens retornados (antes de filtros locais): {len(items)}"
    )

    if not items:
        return [], None

    ref_date = reference_datetime.date()
    candidates: List[Dict[str, Any]] = []

    for item in items:
        dt = item_datetime(item)
        dt_date = dt.date()
        delta_days = abs((dt_date - ref_date).days)

        cloud = get_cloud_cover(item)
        if cloud is None or cloud >= max_cloud:
            continue

        frac = coverage_fraction(
            folha_geom_geojson=folha_geom_geojson,
            item_geom_geojson=item.geometry,
            transformer=transformer,
        )
        if frac < min_coverage:
            continue

        candidates.append(
            {
                "id": item.id,
                "datetime": dt.isoformat(),
                "delta_days": delta_days,
                "coverage_fraction": frac,
                "coverage_percent": frac * 100.0,
                "cloud_cover": cloud,
                "item": item,
            }
        )

    if not candidates:
        print(
            "[S2] Nenhum item atendeu a:\n"
            f"  coverage >= {min_coverage*100:.1f}% e cloud < {max_cloud:.2f}%."
        )
        return [], None

    candidates.sort(
        key=lambda d: (d["delta_days"], d["cloud_cover"], -d["coverage_fraction"])
    )

    if max_print_candidates > 0:
        print(
            f"\n[S2] Itens candidatos (ordenados) "
            f"(primeiros {max_print_candidates} de {len(candidates)}):"
        )
        for c in candidates[:max_print_candidates]:
            print(
                f"  id={c['id']}, "
                f"datetime={c['datetime']}, "
                f"delta_days={c['delta_days']}, "
                f"coverage={c['coverage_percent']:.2f}%, "
                f"cloud={c['cloud_cover']:.2f}%"
            )

    best = candidates[0]
    print(
        "\n[S2] Melhor item:\n"
        f"  id={best['id']}\n"
        f"  datetime={best['datetime']}\n"
        f"  delta_days={best['delta_days']}\n"
        f"  coverage={best['coverage_percent']:.2f}%\n"
        f"  cloud={best['cloud_cover']:.2f}%"
    )
    return candidates, best


# ----------------------------------------------------------------------
# Pipeline de descoberta (sem download/recorte)
# ----------------------------------------------------------------------
def discover_aster_and_s2_for_folha(
    codigo_folha: str,
    aster_target_date_str: str = "2008-07-15",
    aster_search_datetime: str = "2000-01-01/2015-12-31",
    aster_min_coverage: float = 0.90,
    aster_max_cloud: Optional[float] = None,
    s2_search_datetime: str = "2015-06-23/2100-01-01",
    s2_min_coverage: float = 0.90,
    s2_max_cloud: float = 1.0,
    use_aster_date_as_ref_for_s2: bool = True,
) -> Dict[str, Any]:
    """
    Retorna candidatos ASTER e Sentinel-2 com métricas de nuvem/cobertura,
    mas não realiza download de rasters.
    """

    # 1) ASTER
    aster_candidates, aster_best = search_aster_for_folha(
        codigo_folha=codigo_folha,
        target_date_str=aster_target_date_str,
        collection_id="aster-l1t",
        stac_url=PC_STAC_URL,
        search_datetime=aster_search_datetime,
        min_coverage=aster_min_coverage,
        max_cloud=aster_max_cloud,
        max_items=2000,
    )

    # 2) Referência temporal para Sentinel-2
    if use_aster_date_as_ref_for_s2 and aster_best is not None:
        ref_dt_s2 = item_datetime(aster_best["item"])
        print(
            f"\n[PIPELINE] Referência S2 (folha {codigo_folha}) = data da ASTER selecionada: "
            f"{ref_dt_s2.isoformat()}"
        )
    else:
        ref_dt_s2 = parse_target_date(aster_target_date_str)
        print(
            f"\n[PIPELINE] Referência S2 (folha {codigo_folha}) = data alvo aerogeofísica: "
            f"{ref_dt_s2.isoformat()}"
        )

    # 3) Sentinel-2 (sempre roda, independentemente de ASTER ter candidato ou não)
    s2_candidates, s2_best = search_sentinel2_for_folha(
        codigo_folha=codigo_folha,
        reference_datetime=ref_dt_s2,
        collection_id="sentinel-2-l2a",
        stac_url=PC_STAC_URL,
        search_datetime=s2_search_datetime,
        min_coverage=s2_min_coverage,
        max_cloud=s2_max_cloud,
        max_items=2000,
        max_print_candidates=0,  # não listar todos os candidatos S2
    )

    return {
        "aster_candidates": aster_candidates,
        "aster_best": aster_best,
        "s2_candidates": s2_candidates,
        "s2_best": s2_best,
    }


# ----------------------------------------------------------------------
# Main: descoberta contínua para todas as folhas 25k SB21_ZA_II*
# ----------------------------------------------------------------------
def main():
    # 0) Diretório de saída para os CSVs
    out_dir = "./candidatos_orbitais"
    os.makedirs(out_dir, exist_ok=True)

    # 1) Lista de folhas alvo (25k, código começando com SB21_ZA_II)
    codigos_folhas = listar_folhas_25k_sb21_za_ii(PG_CONN_STR, prefixo="SB21_ZA_II2_NW")
    print(f"Encontradas {len(codigos_folhas)} folhas 25k com 'SB21_ZA_II2_NW' no código:")
    for c in codigos_folhas:
        print(f"  - {c}")

    if not codigos_folhas:
        print("Nenhuma folha encontrada. Encerrando.")
        return

    # 2) Parâmetros globais de busca
    aster_target_date_str = "2008-07-15"
    aster_search_datetime = "2000-01-01/2025-12-31"
    s2_search_datetime = "2015-06-23/2100-01-01"

    # ASTER mais permissivo: menor cobertura mínima, sem filtro de nuvem
    aster_min_coverage = 0.80
    aster_max_cloud = None      # não filtrar por nuvem em ASTER

    # Sentinel-2: mantém filtros restritos
    s2_min_coverage = 0.90
    s2_max_cloud = 1.0          # % nuvem S2

    all_aster_rows: List[Dict[str, Any]] = []
    all_s2_rows: List[Dict[str, Any]] = []

    # 3) Loop nas folhas
    for codigo_folha in codigos_folhas:
        print("\n" + "=" * 80)
        print(f"[PIPELINE] Processando folha: {codigo_folha}")
        print("=" * 80)

        res = discover_aster_and_s2_for_folha(
            codigo_folha=codigo_folha,
            aster_target_date_str=aster_target_date_str,
            aster_search_datetime=aster_search_datetime,
            aster_min_coverage=aster_min_coverage,
            aster_max_cloud=aster_max_cloud,
            s2_search_datetime=s2_search_datetime,
            s2_min_coverage=s2_min_coverage,
            s2_max_cloud=s2_max_cloud,
            use_aster_date_as_ref_for_s2=True,
        )

        # ASTER -> linhas para DataFrame global
        for cand in res["aster_candidates"]:
            row = {k: v for k, v in cand.items() if k != "item"}
            row["folha_codigo"] = codigo_folha
            all_aster_rows.append(row)

        # Sentinel-2 -> linhas para DataFrame global
        for cand in res["s2_candidates"]:
            row = {k: v for k, v in cand.items() if k != "item"}
            row["folha_codigo"] = codigo_folha
            all_s2_rows.append(row)

    # 4) Construir DataFrames globais e salvar CSVs
    if all_aster_rows:
        df_aster_all = pd.DataFrame(all_aster_rows)
        out_aster = os.path.join(out_dir, "aster_sb21_za_ii_25k_candidates.csv")
        df_aster_all.to_csv(out_aster, index=False)
        print(f"\n[RESUMO] ASTER: {len(df_aster_all)} linhas salvas em: {out_aster}")
    else:
        print("\n[RESUMO] Nenhum candidato ASTER encontrado para nenhuma folha.")

    if all_s2_rows:
        df_s2_all = pd.DataFrame(all_s2_rows)
        out_s2 = os.path.join(out_dir, "s2_sb21_za_ii_25k_candidates.csv")
        df_s2_all.to_csv(out_s2, index=False)
        print(f"[RESUMO] S2: {len(df_s2_all)} linhas salvas em: {out_s2}")
    else:
        print("[RESUMO] Nenhum candidato Sentinel-2 encontrado para nenhuma folha.")


if __name__ == "__main__":
    main()


Encontradas 1 folhas 25k com 'SB21_ZA_II2_NW' no código:
  - SB21_ZA_II2_NW

[PIPELINE] Processando folha: SB21_ZA_II2_NW
[ASTER] Folha 'SB21_ZA_II2_NW' carregada. Área WGS84 (aprox): 0.015625 (graus²)
[ASTER] Buscando itens em 'aster-l1t'...
[ASTER] Total de itens retornados (antes de filtros locais): 39

[ASTER] Itens candidatos (ordenados):
  id=AST_L1T_00306012006141203_20150514154927, datetime=2006-06-01T14:12:03.804000+00:00, delta_days=775, coverage=100.00%, cloud=42.00%
  id=AST_L1T_00310202005141145_20150511152438, datetime=2005-10-20T14:11:45.205000+00:00, delta_days=999, coverage=100.00%, cloud=11.00%
  id=AST_L1T_00308172005141156_20150510193125, datetime=2005-08-17T14:11:56.925000+00:00, delta_days=1063, coverage=100.00%, cloud=89.00%
  id=AST_L1T_00308012005141202_20150510135953, datetime=2005-08-01T14:12:02.298000+00:00, delta_days=1079, coverage=93.12%, cloud=1.00%
  id=AST_L1T_00305132005141212_20150509113333, datetime=2005-05-13T14:12:12.600000+00:00, delta_days=1159,